<a href="https://colab.research.google.com/github/EngenhariaSoftwarePUCRS/Inteligencia_Artificial/blob/develop/Trabalho01/Trabalho01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
integrantes = ["Augusto Baldino", "Felipe Freitas", "Isabela Kuser", "Luiza Heller", "Maria Eduarda Maia", "Paola Lopes"]
useful_links = [
    "https://pandas.pydata.org/pandas-docs/version/1.0/user_guide/style.html",
    "https://www.geeksforgeeks.org/how-to-replace-values-in-column-based-on-condition-in-pandas/",
    "https://www.geeksforgeeks.org/how-to-add-header-row-to-a-pandas-dataframe/",
    "https://discuss.streamlit.io/t/is-it-possible-to-center-data-on-cell-like-excel/32045/7",
]

In [4]:
# This block contains all required imports
import pandas as pd
from matplotlib import pyplot
from seaborn.palettes import mpl_palette as colors
from sklearn import metrics, model_selection, neighbors
from typing import Annotated, Literal

In [19]:
# This block contains the dataset initial import and column names
targetClass = "Outcome"
X_GANHOU = 'X Ganhou'
O_GANHOU = 'O Ganhou'
VELHA = 'Deu Velha'
TEM_JOGO = 'Tem Jogo'
columnNames = [
    "Top-Left", "Top-Middle", "Top-Right",
    "Middle-Left", "Middle-Middle", "Middle-Right",
    "Bottom-Left", "Bottom-Middle", "Bottom-Right",
    targetClass,
]
dataset = pd.read_csv(
    "tic-tac-toe.data",
    names=columnNames,
)
dataLinesCount = 958

In [9]:
# This block handles cell logic and field values
# It is used for guaranteeing consistency when comparing diferent cells
cellStrType = Literal['x', 'b', 'o']
cellNumType = Literal[1, 0, -1]
cellEntry = cellStrType | cellNumType

class Cell:
    raw: cellStrType
    value: cellNumType
    string: str
    def __init__(self, raw: cellStrType, value: cellNumType):
        self.raw = raw
        self.value = value
        self.string = ' ' if raw == 'b' else raw

    def __str__(self) -> str:
        if (self.raw == 'b'):
            return ' '
        return self.raw

    def __repr__(self) -> str:
        return f"<Cell {self.raw} | {self.value}>"

    def get_entries(self) -> list[cellEntry]:
        return [self.raw, self.value]

CELL_X = Cell('x', 1)
CELL_O = Cell('o', -1)
CELL_B = Cell('b', 0)

In [10]:
# This block is a function that converts a cell entry to a valid cell
# It is used to guarantee consistency when handling the data values
def to_cell(cell: cellEntry) -> Cell:
    # Python...
    if isinstance(cell, Cell):
        return cell

    for cell_model in [CELL_X, CELL_B, CELL_O]:
        if cell in cell_model.get_entries():
            return cell_model

    raise Exception(f"Invalid entry '{cell}', please check your dataset")

In [40]:
# This block is a function that prints the board in a human-readable format
# It is used mostly for debugging
def print_board(board: list[cellEntry]):
    for i in range(0, 9, 3):
        row_raw = board[i:i+3]
        row_cell = [to_cell(cell) for cell in row_raw]
        row_string = [str(cell) for cell in row_cell]
        row_pretty = " | ".join(row_string)
        print(row_pretty)

In [11]:
# This block is a function that checks wether a player has won the game
# It is used on the initial dataset transformation, to convert the target column appropriately
def has_won(player: Cell, board: list[Cell]) -> bool:
    if len(board) != 9:
        raise ValueError("Board cannot have any different number of rows x columns than 9")

    for i in range(0, 9, 3):
        line = board[i:i+3]
        # print("Line: ", line)
        if all(cell == player for cell in line):
            return True

    for i in range(0, 3):
        column = [board[i], board[i+3], board[i+6]]
        # print("Column: ", column)
        if all(cell == player for cell in column):
            return True

    if board[4] != player:
        return False

    diagonal1 = [board[0], board[8]]
    # print("D1: ", diagonal1)
    if all(cell == player for cell in diagonal1):
        return True

    diagonal2 = [board[2], board[6]]
    # print("D2: ", diagonal2)
    if all(cell == player for cell in diagonal2):
        return True

    return False

In [13]:
# Simply check wether dataset appears to be correct on its initial state
dataset.head(5)

,Top-Left,Top-Middle,Top-Right,Middle-Left,Middle-Middle,Middle-Right,Bottom-Left,Bottom-Middle,Bottom-Right,Outcome
0,x,x,x,x,o,o,x,o,o,positive
1,x,x,x,x,o,o,o,x,o,positive
2,x,x,x,x,o,o,o,o,x,positive
3,x,x,x,x,o,o,o,b,b,positive
4,x,x,x,x,o,o,b,o,b,positive


In [20]:
# This block handles the majority of the dataset's transformation
# It should convert all cells into their numeric counterparts
# It should also adjust the output column to the proper value
for index, row in dataset.iterrows():
    board: list[Cell] = []
    for i, cell_str in enumerate(row.array.tolist()[:-1]):
        columnName = columnNames[i]
        cell = to_cell(cell_str)
        dataset.at[index, columnName] = cell.value
        board.append(cell)

    if row[targetClass] == 'positive':
        targetValue = X_GANHOU
    elif has_won(CELL_O, board):
        targetValue = O_GANHOU
    elif CELL_B.value in board:
        targetValue = TEM_JOGO
    else:
        targetValue = VELHA

    dataset.at[index, targetClass] = targetValue

In [15]:
# Simply check wether dataset appears to have been correctly updated
dataset.head(3)

,Top-Left,Top-Middle,Top-Right,Middle-Left,Middle-Middle,Middle-Right,Bottom-Left,Bottom-Middle,Bottom-Right,Outcome
0,1,1,1,1,-1,-1,1,-1,-1,X Ganhou
1,1,1,1,1,-1,-1,-1,1,-1,X Ganhou
2,1,1,1,1,-1,-1,-1,-1,1,X Ganhou


In [16]:
# Simply check wether dataset appears to have been correctly updated
dataset.tail(3)

,Top-Left,Top-Middle,Top-Right,Middle-Left,Middle-Middle,Middle-Right,Bottom-Left,Bottom-Middle,Bottom-Right,Outcome
955,-1,1,-1,1,-1,1,1,-1,1,Deu Velha
956,-1,1,-1,-1,1,1,1,-1,1,Deu Velha
957,-1,-1,1,1,1,-1,-1,1,1,Deu Velha


In [34]:
# Properly check wether dataset was fully transformed to desired outputs (numerically)
x_ganhou_count = dataset[targetClass].eq(X_GANHOU).sum()
o_ganhou_count = dataset[targetClass].eq(O_GANHOU).sum()
deu_velha_count = dataset[targetClass].eq(VELHA).sum()
tem_jogo_count = dataset[targetClass].eq(TEM_JOGO).sum()

outcomes = {
    X_GANHOU: x_ganhou_count,
    O_GANHOU: o_ganhou_count,
    VELHA: deu_velha_count,
    TEM_JOGO: tem_jogo_count,
}

# Calculate the length of each header for formatting
outcome_length = max(len(outcome) for outcome in outcomes.keys())
count_length = max(len(str(count)) for count in outcomes.values())
count_length = max(count_length, len(" Count "))
percentage_length = len("Percentage")

print(f"| {'Outcome':<{outcome_length}} | {'Count':<{count_length}} | {'Percentage':<{percentage_length}} |")
print(f"|{'-' * (outcome_length + 2)}|{'-' * (count_length + 2)}|{'-' * (percentage_length + 2)}|")
for outcome, count in outcomes.items():
    percentage = (count / dataLinesCount) * 100
    print(f"| {outcome:<{outcome_length}} | {count:<{count_length}} | {percentage:>{percentage_length - 1}.4f}% |")

| Outcome   | Count   | Percentage |
|-----------|---------|------------|
| X Ganhou  | 626     |   65.3445% |
| O Ganhou  | 316     |   32.9854% |
| Deu Velha | 16      |    1.6701% |
| Tem Jogo  | 0       |    0.0000% |


In [41]:
# This block splits the dataset equally for testing across different models

# Define the features (X) and the target variable (outcome)
X = dataset.drop(columns=[targetClass])
y = dataset[targetClass]

# Define the split ratios
split_ratios = {'train': 0.7, 'validation': 0.15, 'test': 0.15}
# Assure that it adds to 1
if sum(split_ratios.values()) != 1.0:
    raise Exception("Ratios must add up to 1.0")
validation_plus_test_ratio = split_ratios['validation'] + split_ratios['test']
test_to_validation_ratio = split_ratios['test'] / validation_plus_test_ratio

# Split the data into training, validation, and test sets
X_train, X_temp, y_train, y_temp = model_selection.train_test_split(X, y, train_size=split_ratios['train'], stratify=y, random_state=42)
X_validation, X_test, y_validation, y_test = model_selection.train_test_split(X_temp, y_temp, test_size=test_to_validation_ratio, stratify=y_temp, random_state=42)

# Print the shapes of the resulting sets to verify the split
# Print the distribution of outcomes in each set
print("Training set shape:", X_train.shape)
print("Training set distribution:")
print(y_train.value_counts(normalize=True))

print("\nValidation set shape:", X_validation.shape)
print("Validation set distribution:")
print(y_validation.value_counts(normalize=True))

print("\nTest set shape:", X_test.shape)
print("Test set distribution:")
print(y_test.value_counts(normalize=True))

Training set shape: (670, 9)
Training set distribution:
Outcome
X Ganhou     0.653731
O Ganhou     0.329851
Deu Velha    0.016418
Name: proportion, dtype: float64

Validation set shape: (144, 9)
Validation set distribution:
Outcome
X Ganhou     0.652778
O Ganhou     0.333333
Deu Velha    0.013889
Name: proportion, dtype: float64

Test set shape: (144, 9)
Test set distribution:
Outcome
X Ganhou     0.652778
O Ganhou     0.326389
Deu Velha    0.020833
Name: proportion, dtype: float64


In [42]:
# This block uses the validation set to check wether the dataset is correct
for i in range(len(X_validation)):
    print("Linha", i, "Tabuleiro:")
    print_board(X_validation.iloc[i].values)
    print("Outcome: ", y_validation.iloc[i], end="\n\n")

Linha 0 Tabuleiro:
x |   |  
x | x | o
o | o | x
Outcome:  X Ganhou

Linha 1 Tabuleiro:
x | x | x
  | x | o
o |   | o
Outcome:  X Ganhou

Linha 2 Tabuleiro:
x |   |  
x |   | o
x |   | o
Outcome:  X Ganhou

Linha 3 Tabuleiro:
x | o | x
  | o | x
o |   | x
Outcome:  X Ganhou

Linha 4 Tabuleiro:
x | x | x
o |   | x
  | o | o
Outcome:  X Ganhou

Linha 5 Tabuleiro:
o | x |  
o | o | x
x | x | o
Outcome:  O Ganhou

Linha 6 Tabuleiro:
o | o | o
x |   | x
  | x |  
Outcome:  O Ganhou

Linha 7 Tabuleiro:
x | o | o
x |   |  
x | x | o
Outcome:  X Ganhou

Linha 8 Tabuleiro:
x |   | o
x | x |  
x | o | o
Outcome:  X Ganhou

Linha 9 Tabuleiro:
  | x | x
  | x | o
x | o | o
Outcome:  X Ganhou

Linha 10 Tabuleiro:
x |   |  
o | x | o
x | o | x
Outcome:  X Ganhou

Linha 11 Tabuleiro:
x | o | x
x | o | o
x |   |  
Outcome:  X Ganhou

Linha 12 Tabuleiro:
x | x | x
  |   | o
  |   | o
Outcome:  X Ganhou

Linha 13 Tabuleiro:
x |   | o
o | x |  
x | o | x
Outcome:  X Ganhou

Linha 14 Tabuleiro:
  | x | o


In [39]:
# This block sets up the kNN algorithm and tests its accuracy
kNN_classifier = neighbors.KNeighborsClassifier(n_neighbors=5)
kNN_classifier.fit(X_train, y_train)
predictions_test = kNN_classifier.predict(X_test)
print(f"k-NN accuracy: {metrics.accuracy_score(y_test, predictions_test):.4f}")

k-NN accuracy: 0.9722


In [49]:
# This last block can be used as a "playground" to test the models
input_raw: list[cellEntry] = [
    'o', 'x', 'o',
    'b', 'b', 'x',
    'x', 'x', 'o',
]
print("Entrada: ")
print_board(input_raw)
input_cell = [to_cell(cell) for cell in input_raw]
input_numeric = [cell.value for cell in input_cell]
input = {columnNames[i]: input_numeric[i] for i in range(len(input_numeric))}
input = pd.DataFrame([input])
outcome, *_ = kNN_classifier.predict(input)
print("Saída kNN: ", outcome)

Entrada: 
o | x | o
  |   | x
x | x | o
Saída kNN:  O Ganhou
